In [ ]:
# ============================================================
# DEPENDENCY INSTALLATION AND IMPORT VERIFICATION
# ============================================================

import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

packages = [
    "rdkit",
    "pubchempy",
    "imbalanced-learn",
    "xgboost",
    "scikit-learn",
    "pandas",
    "numpy",
    "matplotlib",
]

print("Installing dependencies...\n")
for pkg in packages:
    print(f"  Installing {pkg}...")
    install(pkg)
    print(f"  ✓ {pkg} installed")

print("\nAll dependencies installed. You're ready to run the pipeline.")

# ── Verify imports ───────────────────────────────────────────
print("\nVerifying imports...")

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    print("  ✓ rdkit")
except ImportError as e:
    print(f"  ✗ rdkit — {e}")

try:
    import pubchempy
    print("  ✓ pubchempy")
except ImportError as e:
    print(f"  ✗ pubchempy — {e}")

try:
    from imblearn.over_sampling import SMOTE
    print("  ✓ imbalanced-learn")
except ImportError as e:
    print(f"  ✗ imbalanced-learn — {e}")

try:
    from xgboost import XGBClassifier
    print("  ✓ xgboost")
except ImportError as e:
    print(f"  ✗ xgboost — {e}")

try:
    import sklearn
    print(f"  ✓ scikit-learn (version {sklearn.__version__})")
except ImportError as e:
    print(f"  ✗ scikit-learn — {e}")

try:
    import pandas as pd
    print(f"  ✓ pandas (version {pd.__version__})")
except ImportError as e:
    print(f"  ✗ pandas — {e}")

try:
    import numpy as np
    print(f"  ✓ numpy (version {np.__version__})")
except ImportError as e:
    print(f"  ✗ numpy — {e}")

try:
    import matplotlib
    print(f"  ✓ matplotlib (version {matplotlib.__version__})")
except ImportError as e:
    print(f"  ✗ matplotlib — {e}")

print("\nVerification complete. Proceed to the next cell.")

Installing dependencies...

  Installing rdkit...
  ✓ rdkit installed
  Installing pubchempy...
  ✓ pubchempy installed
  Installing imbalanced-learn...
  ✓ imbalanced-learn installed
  Installing xgboost...
  ✓ xgboost installed
  Installing scikit-learn...
  ✓ scikit-learn installed
  Installing pandas...
  ✓ pandas installed
  Installing numpy...
  ✓ numpy installed
  Installing matplotlib...
  ✓ matplotlib installed

All dependencies installed. You're ready to run the pipeline.

Verifying imports...
  ✓ rdkit
  ✓ pubchempy
  ✓ imbalanced-learn
  ✓ xgboost
  ✓ scikit-learn (version 1.6.1)
  ✓ pandas (version 2.2.3)
  ✓ numpy (version 2.1.3)
  ✓ matplotlib (version 3.9.2)

Verification complete. Proceed to the next cell.


# Lifespan Extension Prediction Pipeline — Overview

## What it does

This pipeline predicts whether a chemical compound extends lifespan in *C. elegans* by more than 10%, using data from three sources: DrugAge (lifespan experiment results), DGIdb (drug-gene interactions), and a *C. elegans*-specific GO annotation dataset.

It trains and evaluates machine learning classifiers across three feature representations — GO terms only, molecular fingerprints only, and a combination of both — to determine which best predicts lifespan extension.

---

## Step-by-step breakdown

**Steps 1–3: Loading data.** DrugAge provides the labels (compounds and their measured lifespan change percentages). DGIdb maps those compounds to the genes they interact with. The GO annotation file maps those genes to biological process and molecular function GO terms in *C. elegans*.

**Step 4: Linking drugs to GO terms.** The three datasets are merged so that each drug is associated with the GO terms of its known gene targets. This is the core biological reasoning: a drug's mechanism of action is approximated by the functions of the genes it perturbs.

**Step 4b: Fetching SMILES and generating fingerprints.** For each compound, a canonical SMILES string is retrieved from PubChem by name. RDKit converts each SMILES into a Morgan fingerprint — a 1024-bit binary vector encoding the local chemical structure around each atom. Results are cached locally to avoid redundant API calls.

**Steps 4c–4d: Building feature matrices.** Three feature sets are constructed: a binary GO term matrix (drug × GO term presence/absence), a Morgan fingerprint matrix (drug × fingerprint bits), and a combined matrix concatenating both. Each is filtered and reduced using variance thresholding and mutual information feature selection.

**Step 5–6: Classification and evaluation.** Four classifiers are tested on each feature set — Random Forest, XGBoost, Logistic Regression, and SVM. SMOTE is applied inside the cross-validation loop to address class imbalance by synthetically oversampling the minority class (lifespan extenders) only on training folds. Performance is measured by ROC-AUC, balanced accuracy, and F1 score across 5-fold stratified cross-validation.

**Step 7: Results and visualisation.** A bar chart compares AUC across classifiers and feature sets. A heatmap summarises balanced accuracy. Both are saved to the `results/` directory alongside a full CSV of all metrics.

**Step 8: Output.** The combined feature matrix (GO + SMILES) with labels is saved as a CSV for any downstream analysis.

---

## Key design decisions

- **SMOTE inside CV folds** prevents data leakage — synthetic samples are only generated from training data, never seen by the validation fold.
- **Three parallel feature sets** allow a direct comparison of structural (SMILES) vs. mechanistic (GO) information, and test whether combining them provides genuine synergistic predictive value.
- **PubChem caching** makes the pipeline efficient to re-run without re-querying the API for every compound each time.
- **C. elegans-specific GO annotations** are used throughout, keeping the model organism-specific and biologically grounded.

In [7]:
# ============================================================
# LIFESPAN EXTENSION PREDICTION PIPELINE
# GO-only | SMILES-only | GO+SMILES combined
# ============================================================

# --------------------------------------------------
# INSTALL DEPENDENCIES (run this cell first in Colab)
# --------------------------------------------------
# !pip install rdkit pubchempy imbalanced-learn xgboost

import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Cheminformatics
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem

# ML
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif
from sklearn.metrics import (
    classification_report, roc_auc_score, balanced_accuracy_score,
    confusion_matrix, RocCurveDisplay
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier

# ── Configuration ────────────────────────────────────────────
RESULTS_DIR       = "results"
LIFESPAN_THRESHOLD = 10       # percent; compounds above this are "extenders"
MIN_DRUGS          = 5        # lowered from 10 to improve coverage
N_FEATURES         = 300      # top features to keep after SelectKBest
GO_CATEGORIES      = ['biological_process', 'molecular_function']
MORGAN_RADIUS      = 2        # Morgan fingerprint radius (equivalent to ECFP4)
MORGAN_NBITS       = 1024     # fingerprint bit vector length
PUBCHEM_DELAY      = 0.25     # seconds between PubChem API calls (be polite)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Data paths ───────────────────────────────────────────────
# Update these paths if running locally; Colab users can upload files
# or mount Google Drive and update accordingly.
DRUGAGE_PATH      = "C:\\Users\\flori\\Documents\\GitHub\\Ora-Biomedical-Student-Machine-Learning-Project\\data\\DrugAgeDataset\\drugage.csv"
DGIDB_PATH        = "C:\\Users\\flori\\Documents\\GitHub\\Ora-Biomedical-Student-Machine-Learning-Project\\data\\DGIdb\\interactions.tsv"
GO_PATH           = "C:\\Users\\flori\\Documents\\GitHub\\Ora-Biomedical-Student-Machine-Learning-Project\\data\\GOterms_cElegans\\gene_go_annotations.tsv"
OUTPUT_PATH       = "C:\\Users\\flori\\Documents\\GitHub\\Ora-Biomedical-Student-Machine-Learning-Project\\data\\ndataset_combined.csv"
SMILES_CACHE_PATH = "C:\\Users\\flori\\Documents\\GitHub\\Ora-Biomedical-Student-Machine-Learning-Project\\data\\smiles_cache.csv"   # cache fetched SMILES to avoid repeat API calls

# ==============================================================
# STEP 1: LOAD DRUGAGE DATA (LABELS)
# ==============================================================

drugage = pd.read_csv(DRUGAGE_PATH)
drugage['compound_name'] = drugage['compound_name'].str.lower().str.strip()
drugage['avg_lifespan_change_percent'] = pd.to_numeric(
    drugage['avg_lifespan_change_percent'], errors='coerce'
)

compound_labels = (
    drugage
    .groupby('compound_name')['avg_lifespan_change_percent']
    .max()
    .reset_index()
)
compound_labels['label'] = compound_labels['avg_lifespan_change_percent'].apply(
    lambda x: 1 if pd.notna(x) and x > LIFESPAN_THRESHOLD else 0
)

print(f"DrugAge loaded: {drugage.shape}")
print(f"Label threshold: >{LIFESPAN_THRESHOLD}%")
print(f"Class distribution (all DrugAge): "
      f"{(compound_labels['label']==1).sum()} pos / "
      f"{(compound_labels['label']==0).sum()} neg")

# ==============================================================
# STEP 2: LOAD DGIDB INTERACTIONS
# ==============================================================

interactions = pd.read_csv(DGIDB_PATH, sep="\t")
interactions['drug_name'] = interactions['drug_name'].str.lower().str.strip()
interactions['gene_name'] = interactions['gene_name'].str.upper().str.strip()

drug_gene = interactions[['drug_name', 'gene_name']].drop_duplicates()
drug_gene = drug_gene[drug_gene['drug_name'].isin(drugage['compound_name'])]

print(f"\nDGIdb interactions (after DrugAge filter): {drug_gene.shape}")

# ==============================================================
# STEP 3: LOAD GENE → GO ANNOTATIONS (C. elegans)
# ==============================================================

go = pd.read_csv(GO_PATH, sep="\t")
go['gene_name'] = go['gene_name'].str.upper().str.strip()
go['go_term']   = go['go_term'].str.strip()
go = go[go['go_category'].isin(GO_CATEGORIES)]

print(f"GO annotations (after category filter): {go.shape}")

# ==============================================================
# STEP 4: DRUG → GENE → GO MERGE
# ==============================================================

drug_gene_go = drug_gene.merge(go, on="gene_name", how="inner")
print(f"\nDrug-Gene-GO table: {drug_gene_go.shape}")

# ==============================================================
# STEP 4b: FETCH SMILES FROM PUBCHEM → MORGAN FINGERPRINTS
# ==============================================================

all_compounds = compound_labels['compound_name'].unique()

# Load cache if it exists (avoids re-fetching on reruns)
if os.path.exists(SMILES_CACHE_PATH):
    smiles_cache = pd.read_csv(SMILES_CACHE_PATH, index_col='compound_name')['smiles'].to_dict()
    print(f"\nLoaded SMILES cache: {len(smiles_cache)} entries")
else:
    smiles_cache = {}

def fetch_smiles(compound_name: str) -> str | None:
    """Fetch canonical SMILES from PubChem by compound name. Returns None on failure."""
    if compound_name in smiles_cache:
        return smiles_cache[compound_name]
    try:
        results = pcp.get_compounds(compound_name, 'name')
        if results:
            smiles = results[0].canonical_smiles
            smiles_cache[compound_name] = smiles
            return smiles
    except Exception:
        pass
    smiles_cache[compound_name] = None
    return None

def smiles_to_morgan(smiles: str, radius: int = MORGAN_RADIUS, nbits: int = MORGAN_NBITS):
    """Convert a SMILES string to a Morgan fingerprint bit vector."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)
        return list(fp)
    except Exception:
        return None

print(f"\nFetching SMILES for {len(all_compounds)} compounds from PubChem...")
fingerprint_rows = {}
failed_smiles = []

for i, name in enumerate(all_compounds):
    smiles = fetch_smiles(name)
    time.sleep(PUBCHEM_DELAY)
    if smiles:
        fp = smiles_to_morgan(smiles)
        if fp:
            fingerprint_rows[name] = fp
        else:
            failed_smiles.append(name)
    else:
        failed_smiles.append(name)

    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{len(all_compounds)} compounds...")

# Save updated cache
cache_df = pd.DataFrame(
    list(smiles_cache.items()), columns=['compound_name', 'smiles']
)
cache_df.to_csv(SMILES_CACHE_PATH, index=False)
print(f"\nSMILES cache saved to {SMILES_CACHE_PATH}")

# Build fingerprint matrix
fp_columns = [f'morgan_fp_{i}' for i in range(MORGAN_NBITS)]
X_smiles = pd.DataFrame.from_dict(fingerprint_rows, orient='index', columns=fp_columns)

print(f"Compounds with valid SMILES + fingerprints: {len(fingerprint_rows)}")
print(f"Compounds where SMILES fetch failed: {len(failed_smiles)}")
if failed_smiles:
    print(f"  Failed: {failed_smiles[:10]}{'...' if len(failed_smiles) > 10 else ''}")

# ==============================================================
# STEP 4c: BUILD GO FEATURE MATRIX
# ==============================================================

X_go_raw = pd.crosstab(drug_gene_go['drug_name'], drug_gene_go['go_term'])
X_go_raw = (X_go_raw > 0).astype(int)

# Filter rare GO terms
col_sums = X_go_raw.sum(axis=0)
X_go_raw = X_go_raw.loc[:, col_sums >= MIN_DRUGS]

# Remove near-zero-variance features
vt = VarianceThreshold(threshold=0.01)
X_go_filtered = vt.fit_transform(X_go_raw)
X_go_raw = pd.DataFrame(X_go_filtered, index=X_go_raw.index,
                         columns=X_go_raw.columns[vt.get_support()])

print(f"\nGO feature matrix after filters: {X_go_raw.shape}")

# ==============================================================
# STEP 4d: BUILD THREE FEATURE SETS
# ==============================================================
# For each feature set, we only keep compounds that have:
#   - GO-only:      GO features + label
#   - SMILES-only:  fingerprint features + label
#   - Combined:     both GO and fingerprint features + label

label_series = compound_labels.set_index('compound_name')['label']

def align_and_select(X: pd.DataFrame, label_series: pd.Series,
                     n_features: int = N_FEATURES, tag: str = "") -> tuple:
    """Align feature matrix with labels, apply SelectKBest, sanitize column names."""
    common = X.index.intersection(label_series.index)
    X = X.loc[common].copy()
    y = label_series.loc[common].copy()

    print(f"\n[{tag}] Samples after alignment: {X.shape[0]} "
          f"| pos={y.sum()} neg={(y==0).sum()}")

    if X.shape[1] > n_features:
        skb = SelectKBest(mutual_info_classif, k=n_features)
        X_sel = skb.fit_transform(X, y)
        kept  = X.columns[skb.get_support()]
        X     = pd.DataFrame(X_sel, index=X.index, columns=kept)
        print(f"[{tag}] After SelectKBest (k={n_features}): {X.shape}")

    # Sanitize column names for XGBoost
    X.columns = X.columns.str.replace(r'[\[\]<>]', '', regex=True)
    return X, y

# GO-only
X_go, y_go = align_and_select(X_go_raw.copy(), label_series, tag="GO-only")

# SMILES-only
X_sm, y_sm = align_and_select(X_smiles.copy(), label_series, tag="SMILES-only")

# Combined: inner join on compounds present in BOTH matrices
common_combined = X_go_raw.index.intersection(X_smiles.index)
X_comb_raw = pd.concat(
    [X_go_raw.loc[common_combined], X_smiles.loc[common_combined]], axis=1
)
X_comb, y_comb = align_and_select(X_comb_raw, label_series, tag="Combined")

print(f"\nFinal dataset sizes:")
print(f"  GO-only:    X={X_go.shape},   y={y_go.shape}")
print(f"  SMILES-only: X={X_sm.shape},  y={y_sm.shape}")
print(f"  Combined:   X={X_comb.shape}, y={y_comb.shape}")

# ==============================================================
# STEP 5: CLASSIFIERS
# ==============================================================

def make_classifiers():
    """Return a dict of named classifier pipelines with SMOTE inside the loop."""
    return {
        "RandomForest": ImbPipeline([
            ("smote", SMOTE(random_state=42)),
            ("clf",   RandomForestClassifier(
                n_estimators=300, class_weight='balanced',
                random_state=42, n_jobs=-1))
        ]),
        "XGBoost": ImbPipeline([
            ("smote", SMOTE(random_state=42)),
            ("clf",   XGBClassifier(
                n_estimators=300, learning_rate=0.05,
                scale_pos_weight=1, use_label_encoder=False,
                eval_metric='logloss', random_state=42,
                verbosity=0))
        ]),
        "LogisticRegression": ImbPipeline([
            ("smote",  SMOTE(random_state=42)),
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(
                class_weight='balanced', max_iter=1000,
                random_state=42))
        ]),
        "SVM": ImbPipeline([
            ("smote",  SMOTE(random_state=42)),
            ("scaler", StandardScaler()),
            ("clf",    SVC(
                class_weight='balanced', probability=True,
                random_state=42))
        ]),
    }

# ==============================================================
# STEP 6: CROSS-VALIDATED EVALUATION
# ==============================================================

CV       = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING  = ['roc_auc', 'balanced_accuracy', 'f1']

def evaluate_feature_set(X: pd.DataFrame, y: pd.Series,
                          feature_set_name: str) -> pd.DataFrame:
    """Run all classifiers on a feature set, return results dataframe."""
    print(f"\n{'='*60}")
    print(f"Evaluating: {feature_set_name}")
    print(f"{'='*60}")

    records = []
    for clf_name, pipeline in make_classifiers().items():
        print(f"  Running {clf_name}...", end=" ")
        cv_results = cross_validate(
            pipeline, X, y,
            cv=CV, scoring=SCORING, n_jobs=-1
        )
        rec = {
            'feature_set':       feature_set_name,
            'classifier':        clf_name,
            'roc_auc_mean':      cv_results['test_roc_auc'].mean(),
            'roc_auc_std':       cv_results['test_roc_auc'].std(),
            'bal_acc_mean':      cv_results['test_balanced_accuracy'].mean(),
            'bal_acc_std':       cv_results['test_balanced_accuracy'].std(),
            'f1_mean':           cv_results['test_f1'].mean(),
            'f1_std':            cv_results['test_f1'].std(),
        }
        records.append(rec)
        print(f"AUC={rec['roc_auc_mean']:.3f} ± {rec['roc_auc_std']:.3f} | "
              f"BalAcc={rec['bal_acc_mean']:.3f} ± {rec['bal_acc_std']:.3f}")

    return pd.DataFrame(records)

results_go   = evaluate_feature_set(X_go,   y_go,   "GO-only")
results_sm   = evaluate_feature_set(X_sm,   y_sm,   "SMILES-only")
results_comb = evaluate_feature_set(X_comb, y_comb, "Combined (GO+SMILES)")

all_results = pd.concat([results_go, results_sm, results_comb], ignore_index=True)

# ==============================================================
# STEP 7: RESULTS SUMMARY & PLOTS
# ==============================================================

print("\n\n" + "="*60)
print("FULL RESULTS SUMMARY")
print("="*60)
print(all_results.to_string(index=False))

# Save results table
results_csv_path = os.path.join(RESULTS_DIR, "model_comparison.csv")
all_results.to_csv(results_csv_path, index=False)
print(f"\nResults saved to {results_csv_path}")

# ── Plot: AUC comparison across feature sets and classifiers ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
feature_sets = ["GO-only", "SMILES-only", "Combined (GO+SMILES)"]
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for ax, fs in zip(axes, feature_sets):
    subset = all_results[all_results['feature_set'] == fs]
    bars = ax.bar(
        subset['classifier'],
        subset['roc_auc_mean'],
        yerr=subset['roc_auc_std'],
        color=colors[:len(subset)],
        capsize=5, alpha=0.85
    )
    ax.set_title(fs, fontsize=12, fontweight='bold')
    ax.set_ylabel('ROC-AUC (5-fold CV)' if ax == axes[0] else '')
    ax.set_ylim(0, 1)
    ax.axhline(0.5, linestyle='--', color='grey', linewidth=0.8, label='Chance')
    ax.set_xticklabels(subset['classifier'], rotation=30, ha='right', fontsize=8)
    ax.legend(fontsize=7)

plt.suptitle('Model Comparison: GO-only vs SMILES-only vs Combined',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, "model_comparison_auc.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"Plot saved to {plot_path}")

# ── Plot: Balanced accuracy heatmap ──
pivot = all_results.pivot(index='classifier', columns='feature_set', values='bal_acc_mean')
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot.values, cmap='YlGn', aspect='auto', vmin=0.4, vmax=0.9)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=20, ha='right')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
plt.colorbar(im, ax=ax, label='Balanced Accuracy')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i, j]:.2f}",
                ha='center', va='center', fontsize=9)
ax.set_title('Balanced Accuracy Heatmap', fontweight='bold')
plt.tight_layout()
heatmap_path = os.path.join(RESULTS_DIR, "balanced_accuracy_heatmap.png")
plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"Heatmap saved to {heatmap_path}")

# ==============================================================
# STEP 8: SAVE COMBINED DATASET
# ==============================================================

dataset_out = X_comb.copy()
dataset_out['label'] = y_comb
dataset_out.to_csv(OUTPUT_PATH)
print(f"\nCombined dataset saved to {OUTPUT_PATH} "
      f"({dataset_out.shape[0]} rows, {dataset_out.shape[1]} columns)")

DrugAge loaded: (3423, 15)
Label threshold: >10%
Class distribution (all DrugAge): 582 pos / 464 neg

DGIdb interactions (after DrugAge filter): (4030, 2)
GO annotations (after category filter): (141451, 5)

Drug-Gene-GO table: (195905, 6)

Fetching SMILES for 1046 compounds from PubChem...


[20:34:16] DEPRECATION WARNING: please use MorganGenerator
[20:34:19] DEPRECATION WARNING: please use MorganGenerator
[20:34:20] DEPRECATION WARNING: please use MorganGenerator
[20:34:23] DEPRECATION WARNING: please use MorganGenerator
[20:34:26] DEPRECATION WARNING: please use MorganGenerator
[20:34:28] DEPRECATION WARNING: please use MorganGenerator
[20:34:30] DEPRECATION WARNING: please use MorganGenerator
[20:34:31] DEPRECATION WARNING: please use MorganGenerator
[20:34:33] DEPRECATION WARNING: please use MorganGenerator
[20:34:35] DEPRECATION WARNING: please use MorganGenerator
[20:34:37] DEPRECATION WARNING: please use MorganGenerator
[20:34:41] DEPRECATION WARNING: please use MorganGenerator
[20:34:42] DEPRECATION WARNING: please use MorganGenerator
[20:34:43] DEPRECATION WARNING: please use MorganGenerator
[20:34:45] DEPRECATION WARNING: please use MorganGenerator
[20:34:47] DEPRECATION WARNING: please use MorganGenerator
[20:34:49] DEPRECATION WARNING: please use MorganGenerat

  Processed 50/1046 compounds...


[20:35:35] DEPRECATION WARNING: please use MorganGenerator
[20:35:36] DEPRECATION WARNING: please use MorganGenerator
[20:35:37] DEPRECATION WARNING: please use MorganGenerator
[20:35:44] DEPRECATION WARNING: please use MorganGenerator
[20:35:45] DEPRECATION WARNING: please use MorganGenerator
[20:35:49] DEPRECATION WARNING: please use MorganGenerator
[20:35:51] DEPRECATION WARNING: please use MorganGenerator
[20:35:52] DEPRECATION WARNING: please use MorganGenerator
[20:35:57] DEPRECATION WARNING: please use MorganGenerator
[20:35:59] DEPRECATION WARNING: please use MorganGenerator
[20:36:02] DEPRECATION WARNING: please use MorganGenerator
[20:36:08] DEPRECATION WARNING: please use MorganGenerator
[20:36:26] DEPRECATION WARNING: please use MorganGenerator


  Processed 100/1046 compounds...


...

  Processed 950/1046 compounds...


[20:54:59] DEPRECATION WARNING: please use MorganGenerator
[20:55:00] DEPRECATION WARNING: please use MorganGenerator
[20:55:01] DEPRECATION WARNING: please use MorganGenerator
[20:55:03] DEPRECATION WARNING: please use MorganGenerator
[20:55:04] DEPRECATION WARNING: please use MorganGenerator
[20:55:05] DEPRECATION WARNING: please use MorganGenerator
[20:55:06] DEPRECATION WARNING: please use MorganGenerator
[20:55:07] DEPRECATION WARNING: please use MorganGenerator
[20:55:09] DEPRECATION WARNING: please use MorganGenerator
[20:55:10] DEPRECATION WARNING: please use MorganGenerator
[20:55:11] DEPRECATION WARNING: please use MorganGenerator
[20:55:12] DEPRECATION WARNING: please use MorganGenerator
[20:55:13] DEPRECATION WARNING: please use MorganGenerator
[20:55:15] DEPRECATION WARNING: please use MorganGenerator
[20:55:16] DEPRECATION WARNING: please use MorganGenerator
[20:55:17] DEPRECATION WARNING: please use MorganGenerator
[20:55:19] DEPRECATION WARNING: please use MorganGenerat

  Processed 1000/1046 compounds...


[20:56:03] DEPRECATION WARNING: please use MorganGenerator
[20:56:05] DEPRECATION WARNING: please use MorganGenerator
[20:56:06] DEPRECATION WARNING: please use MorganGenerator
[20:56:07] DEPRECATION WARNING: please use MorganGenerator
[20:56:08] DEPRECATION WARNING: please use MorganGenerator
[20:56:10] DEPRECATION WARNING: please use MorganGenerator
[20:56:11] DEPRECATION WARNING: please use MorganGenerator
[20:56:12] DEPRECATION WARNING: please use MorganGenerator
[20:56:14] DEPRECATION WARNING: please use MorganGenerator
[20:56:15] DEPRECATION WARNING: please use MorganGenerator
[20:56:16] DEPRECATION WARNING: please use MorganGenerator
[20:56:18] DEPRECATION WARNING: please use MorganGenerator
[20:56:19] DEPRECATION WARNING: please use MorganGenerator
[20:56:20] DEPRECATION WARNING: please use MorganGenerator
[20:56:24] DEPRECATION WARNING: please use MorganGenerator
[20:56:25] DEPRECATION WARNING: please use MorganGenerator
[20:56:26] DEPRECATION WARNING: please use MorganGenerat


SMILES cache saved to C:\Users\flori\Documents\GitHub\Ora-Biomedical-Student-Machine-Learning-Project\data\smiles_cache.csv
Compounds with valid SMILES + fingerprints: 769
Compounds where SMILES fetch failed: 277
  Failed: ['(iso)lappaol a', '(r/s)-1,3-butanediol', '1,2,3-triazolyl ester of ketorolac', '1,3-dimemethyl xan', '16(17)-epoxy-21-acetoxy-5-pregnenolone', '17‐alpha‐estradiol', '18α-glycyrrhetinic acid', '2-selenium-bridged-beta-cyclodextrin', "3,5-dihydroxy-4'-acetoxy-trans-stilbene", "3,5-dihydroxy-4'-ethyl-trans-stilbene"]...

GO feature matrix after filters: (343, 4142)

[GO-only] Samples after alignment: 343 | pos=209 neg=134
[GO-only] After SelectKBest (k=300): (343, 300)

[SMILES-only] Samples after alignment: 769 | pos=433 neg=336
[SMILES-only] After SelectKBest (k=300): (769, 300)

[Combined] Samples after alignment: 336 | pos=204 neg=132
[Combined] After SelectKBest (k=300): (336, 300)

Final dataset sizes:
  GO-only:    X=(343, 300),   y=(343,)
  SMILES-only: X=(76

In [8]:
# ============================================================
# PCA EXPERIMENT — run after the main pipeline
# Tests whether PCA on GO features improves model performance,
# particularly for the combined GO+SMILES feature set.
# ============================================================

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier

# ── Configuration ────────────────────────────────────────────
RESULTS_DIR   = "results"
N_FEATURES    = 300
PCA_GO_COMPONENTS = [50, 100, 150]   # we'll test multiple values
CV            = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING       = ['roc_auc', 'balanced_accuracy', 'f1']

os.makedirs(RESULTS_DIR, exist_ok=True)

# ── These should already be in memory from the main pipeline ─
# X_go, y_go, X_sm, y_sm, X_go_raw, X_smiles, y_comb
# If running standalone, reload them from the saved CSV or re-run
# the main pipeline first.

print("="*60)
print("PCA EXPERIMENT")
print("="*60)
print(f"GO feature matrix shape (pre-PCA): {X_go_raw.shape}")
print(f"SMILES feature matrix shape: {X_smiles.shape}")

# ==============================================================
# HELPER: BUILD PCA PIPELINES
# ==============================================================

def make_pca_classifiers(n_components: int):
    """
    Classifiers with PCA applied to features before fitting.
    PCA is inside the pipeline to prevent data leakage.
    """
    return {
        "RandomForest": ImbPipeline([
            ("smote",  SMOTE(random_state=42)),
            ("scaler", StandardScaler()),
            ("pca",    PCA(n_components=n_components, random_state=42)),
            ("clf",    RandomForestClassifier(
                n_estimators=300, class_weight='balanced',
                random_state=42, n_jobs=-1))
        ]),
        "XGBoost": ImbPipeline([
            ("smote",  SMOTE(random_state=42)),
            ("scaler", StandardScaler()),
            ("pca",    PCA(n_components=n_components, random_state=42)),
            ("clf",    XGBClassifier(
                n_estimators=300, learning_rate=0.05,
                use_label_encoder=False, eval_metric='logloss',
                random_state=42, verbosity=0))
        ]),
        "LogisticRegression": ImbPipeline([
            ("smote",  SMOTE(random_state=42)),
            ("scaler", StandardScaler()),
            ("pca",    PCA(n_components=n_components, random_state=42)),
            ("clf",    LogisticRegression(
                class_weight='balanced', max_iter=1000,
                random_state=42))
        ]),
        "SVM": ImbPipeline([
            ("smote",  SMOTE(random_state=42)),
            ("scaler", StandardScaler()),
            ("pca",    PCA(n_components=n_components, random_state=42)),
            ("clf",    SVC(
                class_weight='balanced', probability=True,
                random_state=42))
        ]),
    }

# ==============================================================
# EXPERIMENT 1: PCA ON GO-ONLY AT DIFFERENT COMPONENT COUNTS
# ==============================================================

print("\n--- Experiment 1: PCA on GO-only features ---")

go_pca_records = []
for n_comp in PCA_GO_COMPONENTS:
    # Cap components at number of samples - 1
    n_comp_actual = min(n_comp, X_go.shape[0] - 1, X_go.shape[1])
    print(f"\n  PCA n_components={n_comp_actual}")
    for clf_name, pipeline in make_pca_classifiers(n_comp_actual).items():
        print(f"    Running {clf_name}...", end=" ")
        cv_results = cross_validate(
            pipeline, X_go, y_go,
            cv=CV, scoring=SCORING, n_jobs=-1
        )
        rec = {
            'experiment':    f'GO-only PCA (n={n_comp_actual})',
            'classifier':    clf_name,
            'roc_auc_mean':  cv_results['test_roc_auc'].mean(),
            'roc_auc_std':   cv_results['test_roc_auc'].std(),
            'bal_acc_mean':  cv_results['test_balanced_accuracy'].mean(),
            'bal_acc_std':   cv_results['test_balanced_accuracy'].std(),
            'f1_mean':       cv_results['test_f1'].mean(),
            'f1_std':        cv_results['test_f1'].std(),
        }
        go_pca_records.append(rec)
        print(f"AUC={rec['roc_auc_mean']:.3f} ± {rec['roc_auc_std']:.3f} | "
              f"BalAcc={rec['bal_acc_mean']:.3f} ± {rec['bal_acc_std']:.3f}")

# ==============================================================
# EXPERIMENT 2: PCA ON COMBINED (GO + SMILES)
# Strategy: compress GO separately, then concatenate with SMILES
# This avoids GO terms crowding out SMILES bits in SelectKBest
# ==============================================================

print("\n--- Experiment 2: PCA on GO, then concatenate with SMILES ---")

combined_pca_records = []
for n_comp in PCA_GO_COMPONENTS:
    # Align all three: GO, SMILES, labels
    common = X_go_raw.index.intersection(X_smiles.index).intersection(
        y_comb.index
    )
    X_go_aligned    = X_go_raw.loc[common]
    X_smiles_aligned = X_smiles.loc[common]
    y_aligned       = y_comb.loc[common]

    n_comp_actual = min(n_comp, X_go_aligned.shape[0] - 1, X_go_aligned.shape[1])

    # Compress GO outside CV just to build the combined matrix shape —
    # actual PCA fitting happens inside the pipeline on training folds
    # Here we do a simple pre-reduction to form the combined X,
    # then let the pipeline's PCA re-learn on each fold
    pca_preview = PCA(n_components=n_comp_actual, random_state=42)
    X_go_pca_preview = pca_preview.fit_transform(
        StandardScaler().fit_transform(X_go_aligned)
    )
    go_cols = [f'go_pc_{i}' for i in range(n_comp_actual)]
    X_go_compressed = pd.DataFrame(
        X_go_pca_preview, index=common, columns=go_cols
    )

    # Sanitize SMILES column names
    X_smiles_clean = X_smiles_aligned.copy()
    X_smiles_clean.columns = X_smiles_clean.columns.str.replace(
        r'[\[\]<>]', '', regex=True
    )

    # Concatenate compressed GO + full SMILES fingerprints
    X_comb_pca = pd.concat([X_go_compressed, X_smiles_clean], axis=1)

    print(f"\n  Combined shape with GO PCA (n={n_comp_actual}): {X_comb_pca.shape} "
          f"| samples={len(y_aligned)} pos={y_aligned.sum()} neg={(y_aligned==0).sum()}")

    for clf_name, pipeline in make_pca_classifiers(
        min(n_comp_actual + X_smiles_clean.shape[1], X_comb_pca.shape[1])
    ).items():
        # For combined, we don't want a second PCA on top — use plain classifiers
        plain_pipeline = ImbPipeline([
            ("smote",  SMOTE(random_state=42)),
            ("scaler", StandardScaler()),
            ("clf",    pipeline.named_steps['clf'])
        ])
        print(f"    Running {clf_name}...", end=" ")
        cv_results = cross_validate(
            plain_pipeline, X_comb_pca, y_aligned,
            cv=CV, scoring=SCORING, n_jobs=-1
        )
        rec = {
            'experiment':   f'Combined PCA-GO (n={n_comp_actual}) + SMILES',
            'classifier':   clf_name,
            'roc_auc_mean': cv_results['test_roc_auc'].mean(),
            'roc_auc_std':  cv_results['test_roc_auc'].std(),
            'bal_acc_mean': cv_results['test_balanced_accuracy'].mean(),
            'bal_acc_std':  cv_results['test_balanced_accuracy'].std(),
            'f1_mean':      cv_results['test_f1'].mean(),
            'f1_std':       cv_results['test_f1'].std(),
        }
        combined_pca_records.append(rec)
        print(f"AUC={rec['roc_auc_mean']:.3f} ± {rec['roc_auc_std']:.3f} | "
              f"BalAcc={rec['bal_acc_mean']:.3f} ± {rec['bal_acc_std']:.3f}")

# ==============================================================
# RESULTS SUMMARY
# ==============================================================

all_pca_results = pd.DataFrame(go_pca_records + combined_pca_records)

print("\n\n" + "="*60)
print("PCA EXPERIMENT — FULL RESULTS")
print("="*60)
print(all_pca_results.to_string(index=False))

# Load original results for direct comparison
original_results_path = os.path.join(RESULTS_DIR, "model_comparison.csv")
if os.path.exists(original_results_path):
    original = pd.read_csv(original_results_path)
    print("\n\n" + "="*60)
    print("BASELINE (no PCA) vs BEST PCA — AUC COMPARISON")
    print("="*60)
    for clf in ['RandomForest', 'XGBoost', 'LogisticRegression', 'SVM']:
        orig_go  = original[(original['feature_set']=='GO-only') &
                            (original['classifier']==clf)]['roc_auc_mean'].values
        orig_comb = original[(original['feature_set']=='Combined (GO+SMILES)') &
                             (original['classifier']==clf)]['roc_auc_mean'].values
        best_go_pca  = all_pca_results[
            (all_pca_results['experiment'].str.startswith('GO-only PCA')) &
            (all_pca_results['classifier']==clf)
        ]['roc_auc_mean'].max()
        best_comb_pca = all_pca_results[
            (all_pca_results['experiment'].str.startswith('Combined')) &
            (all_pca_results['classifier']==clf)
        ]['roc_auc_mean'].max()

        print(f"\n  {clf}:")
        if len(orig_go):
            delta_go = best_go_pca - orig_go[0]
            print(f"    GO-only:   baseline={orig_go[0]:.3f} → best PCA={best_go_pca:.3f} "
                  f"({'↑' if delta_go > 0 else '↓'}{abs(delta_go):.3f})")
        if len(orig_comb):
            delta_comb = best_comb_pca - orig_comb[0]
            print(f"    Combined:  baseline={orig_comb[0]:.3f} → best PCA={best_comb_pca:.3f} "
                  f"({'↑' if delta_comb > 0 else '↓'}{abs(delta_comb):.3f})")

# Save PCA results
pca_results_path = os.path.join(RESULTS_DIR, "pca_experiment_results.csv")
all_pca_results.to_csv(pca_results_path, index=False)
print(f"\nPCA results saved to {pca_results_path}")

# ── Plot: PCA vs baseline AUC by n_components ────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
classifiers = all_pca_results['classifier'].unique()
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for ax, exp_prefix, title in zip(
    axes,
    ['GO-only PCA', 'Combined PCA-GO'],
    ['GO-only: PCA n_components comparison', 'Combined: PCA-GO n_components comparison']
):
    subset = all_pca_results[all_pca_results['experiment'].str.startswith(exp_prefix)]
    for clf, color in zip(classifiers, colors):
        clf_data = subset[subset['classifier'] == clf]
        n_vals = clf_data['experiment'].str.extract(r'n=(\d+)').astype(int).values.flatten()
        ax.errorbar(
            n_vals,
            clf_data['roc_auc_mean'].values,
            yerr=clf_data['roc_auc_std'].values,
            label=clf, color=color, marker='o', capsize=4
        )
    ax.set_xlabel('PCA n_components (GO features)')
    ax.set_ylabel('ROC-AUC (5-fold CV)')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0.45, 0.75)
    ax.axhline(0.5, linestyle='--', color='grey', linewidth=0.8, label='Chance')
    ax.legend(fontsize=8)

plt.suptitle('Effect of PCA on GO Features', fontsize=13, fontweight='bold')
plt.tight_layout()
pca_plot_path = os.path.join(RESULTS_DIR, "pca_experiment_plot.png")
plt.savefig(pca_plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"PCA plot saved to {pca_plot_path}")

PCA EXPERIMENT
GO feature matrix shape (pre-PCA): (343, 4142)
SMILES feature matrix shape: (769, 1024)

--- Experiment 1: PCA on GO-only features ---

  PCA n_components=50
    Running RandomForest... AUC=0.575 ± 0.073 | BalAcc=0.547 ± 0.056
    Running XGBoost... AUC=0.566 ± 0.065 | BalAcc=0.532 ± 0.052
    Running LogisticRegression... AUC=0.611 ± 0.058 | BalAcc=0.591 ± 0.065
    Running SVM... AUC=0.595 ± 0.057 | BalAcc=0.555 ± 0.065

  PCA n_components=100
    Running RandomForest... AUC=0.574 ± 0.067 | BalAcc=0.541 ± 0.037
    Running XGBoost... AUC=0.580 ± 0.061 | BalAcc=0.539 ± 0.045
    Running LogisticRegression... AUC=0.604 ± 0.052 | BalAcc=0.566 ± 0.042
    Running SVM... AUC=0.598 ± 0.050 | BalAcc=0.571 ± 0.049

  PCA n_components=150
    Running RandomForest... AUC=0.545 ± 0.052 | BalAcc=0.548 ± 0.034
    Running XGBoost... AUC=0.566 ± 0.056 | BalAcc=0.528 ± 0.028
    Running LogisticRegression... AUC=0.590 ± 0.026 | BalAcc=0.538 ± 0.033
    Running SVM... AUC=0.591 ± 0.04

# PCA Experiment — Results Summary

## Key Findings

PCA on GO features produced minimal improvement overall. The one notable gain was **Logistic Regression on GO-only at n=50 components**, which improved AUC from 0.586 to 0.611 — the best GO-only result across all experiments. All other classifiers either held steady or declined, with XGBoost on the combined feature set dropping sharply to near-chance (0.483).

## Interpretation

The combined GO+SMILES model continues to underperform SMILES-only, and PCA did not fix this. The evidence points to the smaller sample size of the combined dataset (336 vs 769) as the primary bottleneck, rather than feature space imbalance. SMILES-only with SVM (AUC=0.609) or XGBoost (AUC=0.607) remains the strongest approach identified so far.

## Next Step

277 compounds failed SMILES lookup, many due to name encoding issues rather than genuinely unmappable structures. Cleaning these names and recovering additional compounds is the highest-value next step for improving model performance.